# Models supervised

## Import Module

In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import cross_val_score
from sklearn import svm

## MLPClassifier

#### On va d'abord essayer de faire une version classique avec ce que l'on connaît

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen

clf = MLPClassifier(random_state=1, max_iter=300).fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.

raise SystemExit(0)

In [ ]:
df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

scaler = MinMaxScaler()
matrices = scaler.fit_transform(matrices)

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y, # permet de garder des proportions cohérentes entre les maladies (évite de n'avoir aucune image d'un type ou inversement)
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen

clf = MLPClassifier(random_state=1, max_iter=500, hidden_layer_sizes=(256, 128, 64)).fit(X_train, y_train) # Crée et entraîne le modèle

print("Test 5 examens")
scores = cross_val_score(clf, X, y, cv=5)
print("Score des 5 examens :", scores)
print("Precision moyenne :", scores.mean())

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision

probabilites = clf.predict_proba(X_test)
print(probabilites[:5])

y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.


Test 5 examens


on obtient :
```

Calcul des 5 examens (patientez quelques minutes)...
Scores des 5 examens : [1.         1.         1.         1.         0.99959823]
Précision moyenne : 0.9999196464443552
Accuracy : 1.0
[[9.99970927e-01 5.49619114e-08 1.07761648e-06 2.79401663e-05]
 [8.58051446e-08 9.99962803e-01 3.70486470e-05 6.20944912e-08]
 [9.99999029e-01 5.82697481e-07 2.97454818e-07 9.09543532e-08]
 [1.54482835e-09 9.99988598e-01 1.13995222e-05 6.78334239e-10]
 [9.45347915e-06 5.18671088e-05 9.99926281e-01 1.23980405e-05]]
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]

```

#### Explication

Pour utiliser MLPClassifier, nous avons essayé plusieurs approches différentes.
Tout d'abord, nous avons divisé (split) nos données pour entraîner le modèle sur 80 % et le tester sur les 20 % restants.
Ensuite, nous avons pris nos échantillons et nous les avons entraînés avec plus ou moins de max_iter.
Et pour mesurer la précision de notre modèle, nous affichons le score d'Accuracy et la matrice de confusion.

Après le premier test, nous obtenons un score d'Accuracy de 1 et une matrice complètement faussée.

Après avoir analysé la situation, on se rend compte d'un problème potentiel qui peut en être la cause : nous n'avons pas les mêmes proportions de sujets sains, de cailloux (calculs), de tumeurs et de kystes. Quand nous prenons 80 % des données pour l'entraînement, il se peut que les proportions ne soient pas respectées.
Imaginons qu'au total on ait 1000 images de cailloux mélangées parmi les 12 000 : rien ne nous garantit que le modèle va tirer des cailloux et qu'il va en prendre assez.
Pour cela, nous allons ajouter stratify=y dans le train_test_split, ce qui garantit d'avoir une proportion minimale respectée.

Cela a permis d'obtenir un meilleur score, mais la matrice de confusion était encore faussée :
```
Accuracy : 0.5671593830334191
[[621   0 306   0]
 [  0 800 470   0]
 [  0   0 344   0]
 [  0   0 571   0]]
```
Pourquoi ?
Car on voit que nous avons quasiment toutes les prédictions concentrées dans 1 seul choix. C'est théoriquement ce que l'on veut, mais notre modèle est trop parfait. Il y a 2 possibilités : la première, on a une fuite de données (data leak) et il apprend par cœur les réponses ; l'option 2, le modèle est surchargé en valeurs avec les arrays.

Nous allons donc procéder dans l'ordre pour essayer de régler ces problèmes. On va d'abord regarder au niveau des arrays.
MLPClassifier n'aime pas les grands nombres, et comme les arrays d'images ont des valeurs comprises entre 0 et 255, on va réduire cela avec MinMaxScaler pour normaliser nos données entre 0 et 1.

Comme cela ne règle pas notre problème, on va essayer de configurer les hidden_layer_sizes.
Comme on a énormément de données en entrée et qu'on doit avoir un petit résultat à la sortie (4 classes), cela crée un gros goulot d'étranglement. La normalisation aide beaucoup, mais les hidden_layer_sizes vont permettre de mettre X neurones au début, puis de faire travailler Y neurones sur le résultat des neurones X, ce qui facilite grandement l'apprentissage.

Mais même malgré cela, nous obtenons des matrices parfaites et des scores parfaits comme :
```
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]
```
Nous avons donc testé une dernière analyse en exécutant cross_val_score. Cela va permettre de vraiment savoir si nous avons un coup de chance au tirage ou si le modèle est simplement très performant.
On obtient :
```
Test 5 examens
Score des 5 examens : [1.         1.         1.         1.         0.99959823]
Precision moyenne : 0.9999196464443552
Accuracy : 1.0

[[9.99970927e-01 5.49619114e-08 1.07761648e-06 2.79401663e-05]
 [8.58051446e-08 9.99962803e-01 3.70486470e-05 6.20944912e-08]
 [9.99999029e-01 5.82697481e-07 2.97454818e-07 9.09543532e-08]
 [1.54482835e-09 9.99988598e-01 1.13995222e-05 6.78334239e-10]
 [9.45347915e-06 5.18671088e-05 9.99926281e-01 1.23980405e-05]]
[[ 742    0    0    0]
 [   0 1016    0    0]
 [   0    0  275    0]
 [   0    0    0  457]]
```
On a donc notre réponse : nos pourcentages de certitude sont de l'ordre de 99,996 % et, comme nous faisons un cross_val_score sur 5 examens tournants, on sait de manière mathématique que ce n'est pas dû à de la malchance ou au hasard du tirage. Le modèle exploite bien une faille dans les données.


Pour régler ce problème on va le régler à la source, on va utiliser un système de groupe (voir preprocessing)



In [3]:
df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices


X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles
groupes = df['groupe']

best_accuracy = 0
meilleur_split = 0

for i in range (1, 5):
    gss = GroupShuffleSplit(n_splits=i, test_size=0.2, random_state=1)
    train_idx, test_idx = next(gss.split(X, y, groups=groupes))
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)   # fit SEULEMENT sur train
    X_test = scaler.transform(X_test)
    
    clf = MLPClassifier(random_state=1, max_iter=300, hidden_layer_sizes=(256, 128, 64)).fit(X_train, y_train) # Crée et entraîne le modèle
    
    
    accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )
    
    print("Accuracy :", accuracy) # Print la précision
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        meilleur_split = i

print(f"la meilleur accuracy est avec : {meilleur_split}, pour un score de {best_accuracy}")

y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.


Accuracy : 0.9251585623678646


/home/gabin/Documents/Project/B-SPE-311-LIL-3-1-cvrie-2/Myvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Accuracy : 0.9302325581395349
Accuracy : 0.9251585623678646


/home/gabin/Documents/Project/B-SPE-311-LIL-3-1-cvrie-2/Myvenv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Accuracy : 0.9408033826638478
la meilleur accuracy est avec : 4, pour un score de 0.9408033826638478
[[818   0   0   0]
 [  0 914 129   0]
 [  0  11 214   0]
 [  0   0   0 279]]


#### On obtient alors 
```
Accuracy : 0.9251585623678646
Accuracy : 0.9302325581395349
Accuracy : 0.9251585623678646
Accuracy : 0.9408033826638478
la meilleur accuracy est avec : 4, pour un score de 0.9408033826638478
[[818   0   0   0]
 [  0 914 129   0]
 [  0  11 214   0]
 [  0   0   0 279]]
```

Ce résultat est déjà beaucoup plus satisfaisant que le précédent.

##### Actuellement nous avons une version complète mais nous avançons à l'aveugle, il faut pour avoir une précision cohérente et proche de la réalité essayer plusieurs paramètres de `max_iter` et `hidden_layer_sizes`. Le `n_splits` est mis sur 5 car c'est la norme, la plupart des modèles utilisent un découpage en 5.

##### Pour ce faire on ne va pas utiliser des boucles qui changent les valeurs, car notre modèle commence à être assez lourd pour gérer plusieurs combinaisons de paramètres. Nous allons utiliser `GridSearchCV`, il ne réduira pas forcément le temps d'exécution mais va répartir la charge sur tous les cœurs/threads du processeur et alléger la charge. Il nous donnera parfaitement le meilleur résultat avec nos paramètres.

In [4]:
df = pd.read_csv("../../Supervise.csv") 
matrices = np.load('../../Matrice.npy', allow_pickle=True)

# classique data
X = matrices 
y = df["target"]
groupes = df['groupe']

# on crée une pipeline qui va contenir notre MinMaxScaler ainsi que le MLPClassifier
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),                 
    ('mlp', MLPClassifier(random_state=1))      
])

# Nos paramètres à tester pour le modèle
param_grid = {
    'mlp__hidden_,layer_sizes': [(128, 64), (256, 128, 64)],
    'mlp__max_iter': [300, 500]
}

# le mélange des groupes tout en faisant attention aux mêmes patients pour le data leak
gss = GroupShuffleSplit(n_splits=3, test_size=0.2, random_state=1)


# utilisation de GridSearchCV pour aller chercher les meilleurs paramètres dans notre modèle
grid_search = GridSearchCV(
    estimator=pipeline,    # chaîne de montage à réaliser
    param_grid=param_grid,    # paramètres qu'on va tester
    cv=gss,    # méthode pour mélanger 
    n_jobs=-1     # nombre de cœurs physiques
)

# on entraîne le modèle pour chercher le meilleur
grid_search.fit(X, y, groups=groupes)

# on affiche nos deux meilleurs paramètres
print("Meilleurs parametres trouves: ", grid_search.best_params_)


ValueError: Invalid parameter 'hidden_,layer_sizes' for estimator MLPClassifier(random_state=1). Valid parameters are: ['activation', 'alpha', 'batch_size', 'beta_1', 'beta_2', 'early_stopping', 'epsilon', 'hidden_layer_sizes', 'learning_rate', 'learning_rate_init', 'max_fun', 'max_iter', 'momentum', 'n_iter_no_change', 'nesterovs_momentum', 'power_t', 'random_state', 'shuffle', 'solver', 'tol', 'validation_fraction', 'verbose', 'warm_start'].

##### Comment ça marche ?
Concrètement on va utiliser un entraînement pour trouver les meilleurs paramètres avec une pipeline.
On va définir nos méthodes dans la pipeline (Normalisation avec MinMaxScaler et notre modèle pour lui trouver les meilleurs paramètres)

On définit ensuite les paramètres à tester ainsi que nos données qui vont servir pour l'entraînement et le test (dans notre cas on coupe en 5 mais on utilise `GroupShuffleSplit` pour ne pas avoir de dataleak entre entraînement et test)

On définit après notre `GridSearchCV`, on lui donne les instructions (celles de la pipeline) puis les paramètres à tester, puis comment on va mélanger puis le nombre de cœurs (-1 utilisant tous les cœurs physiques disponibles)

On peut ensuite entraîner et afficher les meilleurs paramètres avec `best_params_`.


Informations utiles :
    - Pour accéder et utiliser une sous-méthode de notre pipeline il faut mettre sa clé suivie de `__` par exemple si la clé est `test` on doit mettre `test__`


Après exécution avec 
```
param_grid = {
    'mlp__hidden_,layer_sizes': [(128, 64), (256, 128 64)],
    'mlp__max_iter': [300, 500]
}
```

On obtient : `Meilleurs parametres trouves:  {'mlphidden_layer_sizes': (128, 64), 'mlpmax_iter': 300}`.

Nous avons alors notre réponse, nos meilleurs paramètres sont en couche cachée 128 et 64 et en itération 300.

Ceci conclut notre MLPClassifier, nous avons donc un score proche des 99 %, allant de 92 à 99 selon les paramètres (malheureusement les limitations machine ne nous permettent pas d'obtenir un résultat 100% correct.

#### Graphique

#### On va entraîner avec nos valeurs parfaites qu'on a trouvées pour les graphiquesclf = MLPClassifier(random_state=1, max_iter=300, hidden_layer_sizes=(256, 128, 64)).fit(X_train, y_train)

In [1]:
clf = MLPClassifier(random_state=1, max_iter=300, hidden_layer_sizes=(128, 64)).fit(X_train, y_train)

NameError: name 'MLPClassifier' is not defined

In [3]:
y_pred = clf.predict(X_test)

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Création du graphique
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm)

# Noms des classes
classes = clf.classes_

ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))

ax.set_xticklabels(classes)
ax.set_yticklabels(classes)

# Ajouter les nombres dans les cases
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j],
                ha="center",
                va="center")

ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
ax.set_title("Matrice de confusion")

plt.colorbar(im)
plt.tight_layout()
plt.show()

NameError: name 'clf' is not defined

Permet de visualiser graphiquement la matrice

### LogisticRegression

On va utiliser un modèle de LogisticRegression, on va faire une première version.

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen
clf = LogisticRegression()
clf = clf.fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.

raise SystemExit(0)


##### on va directement ajouter nos modifications avec le système de groupe et de normalisation

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles
groupes = df['groupe']


gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=1)
train_idx, test_idx = next(gss.split(X, y, groups=groupes))
    
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


clf = LogisticRegression()
clf = clf.fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)



print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.

raise SystemExit(0)


On obtient ceci :
```
Accuracy : 0.9251585623678646
[[816   2   0   0]
 [ 25 886 132   0]
 [  0  18 207   0]
 [  0   0   0 279]]
```
Le résultat est encore une fois très bon, on va aller l'analyser en profondeur pour le comprendre avec des metrics

In [ ]:
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de classification détaillé :")
print(classification_report(y_test, y_pred))

##### Nous obtenons donc :
```
Matrice de confusion :
[[816   2   0   0]
 [ 25 886 132   0]
 [  0  18 207   0]
 [  0   0   0 279]]

Rapport de classification détaillé :
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       818
           1       0.98      0.85      0.91      1043
           2       0.61      0.92      0.73       225
           3       1.00      1.00      1.00       279

    accuracy                           0.93      2365
   macro avg       0.89      0.94      0.91      2365
weighted avg       0.94      0.93      0.93      2365
```

Cette analyse nous permet d'obtenir plusieurs indications clés.
Premièrement on remarque un déséquilibre on a 1043 images de la maladie 1 contre 300 ou 800 dans support.
Recall c'est sur le nombre d'images combien en % il arrive à parfaitement détecter, 0.85 étant le score le plus bas ce qui est acceptable.

Precision par contre c'est la sûreté de détection (savoir quand elle crie au loup) et elles sont toutes très performantes sauf pour la maladie 2, l'IA se trompe 4 fois sur 6.

La note globale qui est le f1 est tout de même assez bonne, seule la maladie 2 a des difficultés car elle ressemble trop mathématiquement à la maladie 1


On va essayer tout de même de rééquilibrer les proportions comme on a énormément de 1 et que le 2 est moins important.
Pour ce faire on va appliquer un poids équilibré avec `class_weight='balanced'` et lui mettre plus de `max_iter`.
Également on va faire une normalisation avec un MinMaxScaler.

Pourquoi notre modèle fonctionne sans MinMaxScaler ?
Il fonctionne grâce à des proportions partielles, LogisticRegression a besoin de données équilibrées, on ne peut pas lui donner un salaire entre 0 et 100000 et à côté donner l'âge entre 0 et 100 il ne va pas réussir. Mais comme nous utilisons des images nous restons entre 0 et 255 donc avec un certain équilibre. Mais nous allons quand même en effectuer une pour plus de rapidité et de précision.

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles
groupes = df['groupe']


gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=1)
train_idx, test_idx = next(gss.split(X, y, groups=groupes))
    
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # fit SEULEMENT sur train
X_test = scaler.transform(X_test)

clf = LogisticRegression(class_weight='balanced', max_iter=1000)
clf = clf.fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)


print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de classification détaillé :")
print(classification_report(y_test, y_pred))


raise SystemExit(0)


# On obtient :

```
Accuracy : 0.9221987315010571
Matrice de confusion :
[[816   2   0   0]
 [ 30 879 134   0]
 [  0  18 207   0]
 [  0   0   0 279]]

Rapport de classification détaillé :
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       818
           1       0.98      0.84      0.91      1043
           2       0.61      0.92      0.73       225
           3       1.00      1.00      1.00       279

    accuracy                           0.92      2365
   macro avg       0.89      0.94      0.90      2365
weighted avg       0.94      0.92      0.93      2365
```

On voit donc une seule différence, la précision du 01 qui descend de 0.01.
Nos changements n'ont donc aucun effet, on ne peut pas pousser plus loin, nous avons atteint la limite du modèle de LogisticRegression, on ne peut simplement pas tracer une ligne parfaite entre la maladie 01 et 02.

##### Graphique pour visualiser notre résultat final

In [ ]:
y_pred = clf.predict(X_test)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Création du graphique
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm)

# Noms des classes
classes = clf.classes_

ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))

ax.set_xticklabels(classes)
ax.set_yticklabels(classes)

# Ajouter les nombres dans les cases
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j],
                ha="center",
                va="center")

ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
ax.set_title("Matrice de confusion")

plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# on veut 2 coordonnées X et Y
pca = PCA(n_components=2)

#Transforme le vecteur de mes textes en vecteur 3D
points_2d = pca.fit_transform(X_train)

# on crée la figure
fig = plt.figure(figsize=(10, 8))
axe = fig.add_subplot(111)


nuage = axe.scatter(points_2d[:, 0], points_2d[:, 1], c=y_train, cmap='jet', alpha=0.7)


plt.title("Répartition des 4 maladies avec un model de LogisticRegression")
axe.set_xlabel("Dimension visuelle 1")
axe.set_ylabel("Dimension visuelle 2")


plt.colorbar(nuage, label="Numéro de la maladie")

plt.show()



#### Le résultat de notre graphique nous montre parfaitement nos résultats.
Le bleu ciel et le marron sont assez distincts côte à côte sans se chevaucher.
Mais le bleu foncé avec le jaune c'est une autre histoire, on voit que le bleu foncé prend le dessus sur le jaune. C'est le même résultat qu'avec notre f1-score, les deux étant trop proches l'un prend l'ascendant sur l'autre et on détecte mieux et plus souvent du bleu foncé que du jaune (maladie 2) qui est assez proche mathématiquement du bleu foncé (maladie 0).

Ceci explique pourquoi nous avons un score sur la maladie 2 défectueux.

### SVM

In [ ]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=1
) # Split X et Y pour avoir une partie entraînement et une partie examen

clf = svm.SVC().fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot ( le % d'IRM qu'il arrive à classer )

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) # Crée la matrice de précision pour savoir avec plus de précision comment le bot arrive à classer les différentes maladies genre à combien de pourcent il devine que les patients atteints de stone sont atteints de stone.

raise SystemExit(0)


Comme d'habitude le résultat du modèle est sans appel
```
Accuracy : 1.0
[[ 927    0    0    0]
 [   0 1270    0    0]
 [   0    0  344    0]
 [   0    0    0  571]]
```

On va donc ajouter le système de groupe et de normalisation.

In [5]:
try:
    df = pd.read_csv("../../Supervise.csv") # Ouvre le csv
except FileNotFoundError:
    raise SystemExit(84)

matrices = np.load('../../Matrice.npy', allow_pickle=True) # Ouvre les matrices

X = matrices # Donne les matrices en données afin qu'il classe les IRM des patients en fonction de leurs maladies
y = df["target"] # Donne les différentes maladies possibles
groupes = df['groupe']

gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=1)
train_idx, test_idx = next(gss.split(X, y, groups=groupes))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # fit SEULEMENT sur train
X_test = scaler.transform(X_test)
    
clf = svm.SVC().fit(X_train, y_train) # Crée et entraîne le modèle

accuracy = clf.score(X_test, y_test) # Calcule la précision du bot (le % d'IRM qu'il arrive à classer)

print("Accuracy :", accuracy) # Print la précision


y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred)) 

raise SystemExit(0)

Accuracy : 0.9911205073995771
[[ 817    0    0    1]
 [   0 1043    0    0]
 [   0   16  205    4]
 [   0    0    0  279]]


SystemExit: 0

/home/gabin/Documents/Project/B-SPE-311-LIL-3-1-cvrie-2/Myvenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


##### Grpahique d'analyse de la matrice

In [ ]:
y_pred = clf.predict(X_test)


# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_pred)

# Création du graphique
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm)

# Noms des classes
classes = clf.classes_

ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))

ax.set_xticklabels(classes)
ax.set_yticklabels(classes)

# Ajouter les nombres dans les cases
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j],
                ha="center",
                va="center")

ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
ax.set_title("Matrice de confusion")

plt.colorbar(im)
plt.tight_layout()
plt.show()

Nous obtenons :
```
Accuracy : 0.9911205073995771
[[ 817    0    0    1]
 [   0 1043    0    0]
 [   0   16  205    4]
 [   0    0    0  279]]
```

Le résultat est encore une fois très performant, le modèle est sur le papier complet et n'a pas de problème. 
Nous allons tout de même essayer avec plusieurs tolérances de groupes d'images.
Le résultat précédent était avec une ressemblance de 0.995.

Avec 0.9 nous obtenons :
```
Accuracy : 0.5722911774572291
[[579   0 100  15]
 [ 37 872 455 270]
 [ 30  13  74  18]
 [ 74 164  99 181]]
```

On peut donc se demander si notre ressemblance de 0.995 n'est pas fausse. Il faut donc analyser cela pour établir le meilleur. (Voir PreProcessing.ipynb pour les calculs et l'analyse).

## Comparaison et conclusion

Pour conclure, nous avons donc 3 modèles différents avec des résultats finaux assez proches les uns des autres. Pour être conforme au sujet, il faudrait décider du meilleur modèle, mais le choix final dépend véritablement du besoin et de l'utilisation attendue.

Si l'on recherche de la précision pure, il faut se tourner vers le SVM et le MLP, car tous deux règlent le problème de la Régression Logistique sur la maladie 2. Cependant, le MLP serait notre grand gagnant : ses résultats sont bien plus que suffisants et il s'entraîne beaucoup plus rapidement que le SVM, dont le temps de calcul devient extrêmement lourd à mesure que le volume d'images augmente.

À l'inverse, si l'on cherche un modèle léger capable de catégoriser rapidement et de manière explicable, la Régression Logistique serait la gagnante. Son seul point faible réside dans sa gestion de la maladie 2 : comme les caractéristiques des maladies 1 et 2 se chevauchent, le modèle peine à tracer une séparation nette, ce qui limite le score de cette classe spécifique.


## La Loss Function (Fonction de Perte) : Le moteur de l'apprentissage

Contrairement à l'**Accuracy** (qui calcule un pourcentage de réussite compréhensible par l'humain), la **Loss Function** est la métrique interne utilisée par la machine pendant son entraînement. Elle mesure mathématiquement l'écart entre la prédiction du modèle et la vraie réponse (la *target*). 

Le but de chaque algorithme est de **minimiser cette perte**. C'est grâce à ce calcul d'erreur que le modèle sait *comment* corriger ses paramètres internes pour s'améliorer au cycle suivant.

Voici les fonctions de perte utilisées par nos trois modèles supervisés :

### 1. Régression Logistique : La Log-Loss (Entropie Croisée)
* **Le principe :** La Régression Logistique calcule des probabilités (ex: "Je suis sûr à 90% que c'est la maladie 1"). La Log-Loss pénalise lourdement le modèle s'il se trompe tout en étant très confiant. 
* **Le rôle :** Au fur et à mesure de l'entraînement, cette fonction force la ligne de séparation (l'hyperplan) à se placer de manière à ce que les probabilités prédites soient les plus proches possible de 100% pour la bonne classe et de 0% pour les autres.

### 2. Support Vector Machine (SVM) : La Hinge Loss (Perte Charnière)
* **Le principe :** Le SVM ne cherche pas seulement à séparer les maladies, il cherche à créer une "zone de sécurité" (la marge) la plus large possible entre elles. 
* **Le rôle :** La Hinge Loss ne pénalise pas seulement les points mal classés, elle pénalise aussi les points qui sont bien classés mais qui se trouvent *trop près* de la frontière de décision (dans la marge). C'est ce calcul qui permet au SVM d'être si robuste sur les cas complexes.

### 3. MLPClassifier (Réseau de Neurones) : La Cross-Entropy Loss
* **Le principe :** Similaire à la Log-Loss, mais appliquée à l'échelle d'une architecture complexe de couches cachées. 
* **Le rôle :** C'est le cœur du réseau de neurones. À chaque itération (époque), le réseau calcule sa Cross-Entropy Loss globale, puis utilise un mécanisme appelé la **rétropropagation du gradient** (*backpropagation*). Il remonte à travers ses couches (256, puis 128, puis 64 neurones) pour ajuster le "poids" de chaque connexion afin de faire baisser mathématiquement cette perte à l'itération suivante.